# 基于机器学习的情感分析方法

## 1. 学习目标

完成本课程后，你应该能够：

- 理解机器学习情感分析的基本原理
- 掌握情感分析的数据准备方法
- 能够使用多种分类算法进行情感分析
- 了解如何评估和优化情感分析模型
- 能够对比词典方法和机器学习方法

## 2. 为什么要用机器学习？

### 2.1 词典方法的局限


**问题1：词典覆盖不全**
```
"这个产品给力！" 
→ 如果词典中没有"给力"，就无法识别
```

**问题2：无法处理复杂语境**
```
"这部电影很沉重"
→ 对电影来说可能是褒义（有深度）
→ 但词典方法只能简单匹配"沉重"这个词
```

**问题3：规则难以穷尽**
```
否定、程度、转折等规则都需要手动定义
很难覆盖所有语言现象
```

### 2.2 机器学习的优势

**机器学习方法可以自动从数据中学习规律：**

**优势1：自动学习特征**
- 不需要人工定义所有情感词
- 可以发现隐含的情感模式

**优势2：考虑上下文**
- 能理解词语在特定语境下的含义
- 自动捕捉词语之间的关系

**优势3：效果更好**
- 在大多数情况下准确率更高
- 尤其适合复杂文本

### 2.3 机器学习情感分析的基本思路

**核心思想**：把情感分析看作一个文本分类问题

```
情感分析 = 文本分类

输入：文本
输出：情感类别（正面/负面/中性）

方法：
1. 收集带标签的训练数据
2. 提取文本特征（TF-IDF等）
3. 训练分类模型
4. 用模型预测新文本的情感
```

## 3. 数据准备

首先，我们需要准备训练数据。

### 3.1 准备示例数据

我们使用电商评论作为示例数据。

In [ ]:
# 准备训练数据
train_texts = [
    # 正面评论（20条）
    "质量很好，物流很快，非常满意",
    "产品不错，价格实惠，值得购买",
    "包装精美，东西很好，五星好评",
    "性价比高，用起来很方便，推荐",
    "物超所值，卖家服务态度好",
    "收到了，质量不错，很喜欢",
    "宝贝很好，和描述一样，满意",
    "非常棒，超出预期，强烈推荐",
    "东西很好用，质量有保证",
    "很满意的一次购物，赞",
    "产品质量优秀，物流超快",
    "性能强大，外观漂亮，喜欢",
    "用了几天，感觉很好，值得",
    "做工精细，材质不错，好评",
    "效果很好，价格合理，满意",
    "商品完好，物流给力，好",
    "宝贝收到了，很不错，推荐购买",
    "质量可以，用着舒服，满意",
    "非常好的产品，会回购",
    "东西不错，物有所值，好评",
    
    # 负面评论（20条）
    "质量太差，完全不值这个价",
    "用了一天就坏了，非常失望",
    "物流太慢，等了好久才到",
    "和描述不符，感觉被骗了",
    "质量很差，不推荐购买",
    "浪费钱，根本没法用",
    "客服态度差，问题也不解决",
    "收到货就发现有问题，差评",
    "做工粗糙，材质很差",
    "不值这个价，后悔买了",
    "完全是骗人的，太坑了",
    "用了几次就坏了，垃圾",
    "包装破损，东西也不好",
    "根本不能用，退货了",
    "质量有问题，商家态度差",
    "虚假宣传，实物差太多",
    "太失望了，不建议购买",
    "性价比太低，不划算",
    "产品有缺陷，很生气",
    "完全达不到预期，差评"
]

# 对应的标签（0表示负面，1表示正面）
train_labels = [1]*20 + [0]*20

print(f"训练数据数量: {len(train_texts)}")
print(f"正面评论: {train_labels.count(1)}条")
print(f"负面评论: {train_labels.count(0)}条")
print(f"\n正面示例: {train_texts[0]}")
print(f"负面示例: {train_texts[20]}")

### 3.2 准备测试数据

In [ ]:
# 准备测试数据
test_texts = [
    # 正面（5条）
    "这个产品真的很好，强烈推荐给大家",
    "收到货了，质量超出预期，很满意",
    "物流速度快，包装很好，产品也不错",
    "用了几天，感觉很好，值得购买",
    "性价比很高，质量也很好，好评",
    
    # 负面（5条）
    "太差劲了，完全不能用，浪费钱",
    "质量有问题，用了两天就出毛病了",
    "和图片差距太大，感觉被欺骗了",
    "客服态度恶劣，问题也不给解决",
    "根本不值这个价格，非常后悔"
]

test_labels = [1]*5 + [0]*5

print(f"测试数据数量: {len(test_texts)}")
print(f"正面评论: {test_labels.count(1)}条")
print(f"负面评论: {test_labels.count(0)}条")

## 4. 文本预处理

对文本进行分词处理。

In [ ]:
import jieba

# 对训练文本分词
train_texts_seg = [' '.join(jieba.lcut(text)) for text in train_texts]

# 对测试文本分词
test_texts_seg = [' '.join(jieba.lcut(text)) for text in test_texts]

print("文本分词完成！")
print(f"\n原文: {train_texts[0]}")
print(f"分词: {train_texts_seg[0]}")

## 5. 特征提取

使用TF-IDF将文本转换为数字特征。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 创建TF-IDF向量化器
vectorizer = TfidfVectorizer(max_features=500)

# 拟合并转换训练数据
X_train = vectorizer.fit_transform(train_texts_seg)

# 转换测试数据
X_test = vectorizer.transform(test_texts_seg)

# 标签
y_train = train_labels
y_test = test_labels

print(f"特征提取完成！")
print(f"训练集形状: {X_train.shape}")
print(f"测试集形状: {X_test.shape}")
print(f"特征数量: {X_train.shape[1]}")

## 6. 模型训练与评估

我们将尝试多个分类算法，对比它们的效果。

### 6.1 模型1：朴素贝叶斯

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print("训练朴素贝叶斯模型...")

# 创建并训练模型
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 预测
y_pred_nb = nb_model.predict(X_test)

# 评估
accuracy_nb = accuracy_score(y_test, y_pred_nb)

print(f"\n朴素贝叶斯准确率: {accuracy_nb:.2%}")
print("\n详细报告：")
print(classification_report(y_test, y_pred_nb, target_names=['负面', '正面']))

### 6.2 模型2：逻辑回归

In [ ]:
from sklearn.linear_model import LogisticRegression

print("训练逻辑回归模型...")

# 创建并训练模型
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# 预测
y_pred_lr = lr_model.predict(X_test)

# 评估
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print(f"\n逻辑回归准确率: {accuracy_lr:.2%}")
print("\n详细报告：")
print(classification_report(y_test, y_pred_lr, target_names=['负面', '正面']))

### 6.3 模型3：支持向量机（SVM）

In [ ]:
from sklearn.svm import LinearSVC

print("训练SVM模型...")

# 创建并训练模型
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

# 预测
y_pred_svm = svm_model.predict(X_test)

# 评估
accuracy_svm = accuracy_score(y_test, y_pred_svm)

print(f"\nSVM准确率: {accuracy_svm:.2%}")
print("\n详细报告：")
print(classification_report(y_test, y_pred_svm, target_names=['负面', '正面']))

### 6.4 模型对比

In [ ]:
# 对比所有模型
import pandas as pd

results = pd.DataFrame({
    '模型': ['朴素贝叶斯', '逻辑回归', 'SVM'],
    '准确率': [accuracy_nb, accuracy_lr, accuracy_svm]
})

results = results.sort_values('准确率', ascending=False)

print("\n模型性能对比：")
print("="*40)
print(results.to_string(index=False))
print("="*40)

## 7. 模型应用

使用训练好的模型对新评论进行情感分析。

### 7.1 定义预测函数

In [ ]:
def predict_sentiment(text, model, vectorizer):
    """
    预测文本的情感倾向
    """
    # 分词
    text_seg = ' '.join(jieba.lcut(text))
    
    # 特征提取
    text_vec = vectorizer.transform([text_seg])
    
    # 预测
    prediction = model.predict(text_vec)[0]
    sentiment = '正面' if prediction == 1 else '负面'
    
    # 如果模型支持概率预测
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(text_vec)[0]
        confidence = proba[prediction]
    else:
        # SVM使用decision_function
        decision = model.decision_function(text_vec)[0]
        # 将决策值转换为近似概率
        confidence = 1 / (1 + abs(decision))
    
    return {
        'sentiment': sentiment,
        'confidence': confidence,
        'label': prediction
    }

print("✅ 预测函数定义完成！")

### 7.2 测试新评论

In [ ]:
# 新的评论
new_reviews = [
    "这个产品真的超级棒，用起来很舒服，强烈推荐！",
    "太失望了，质量太差，完全不值这个价钱",
    "还行吧，没有特别好，也没有特别差",
    "物流超快，包装很好，东西也不错，满意",
    "垃圾产品，用了一天就坏了，绝对差评"
]

print("新评论情感分析（使用逻辑回归模型）：")
print("="*80)

for review in new_reviews:
    result = predict_sentiment(review, lr_model, vectorizer)
    
    print(f"\n评论: {review}")
    print(f"情感: {result['sentiment']}")
    print(f"置信度: {result['confidence']:.2%}")
    print("-"*80)

## 8. 模型优化技巧

### 8.1 优化方向1：增加训练数据

**最有效的优化方法是增加训练数据量。**

**数据来源：**
- 电商平台评论
- 社交媒体帖子
- 公开数据集
- 人工标注

**注意事项：**
- 保证数据质量
- 标注要准确
- 类别要平衡

### 8.2 优化方向2：特征工程

**可以尝试的特征：**

**1. N-gram特征**
```python
# 使用1-gram和2-gram
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
```

**2. 字符级特征**
```python
# 使用字符n-gram
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
```

**3. 词向量特征**
```python
# 使用Word2Vec或预训练词向量
```

### 8.3 优化方向3：模型调参

**每个模型都有可调整的参数：**

**朴素贝叶斯：**
```python
MultinomialNB(alpha=1.0)  # 平滑参数
```

**逻辑回归：**
```python
LogisticRegression(C=1.0)  # 正则化参数
```

**SVM：**
```python
LinearSVC(C=1.0)  # 惩罚参数
```

### 8.4 优化方向4：集成学习

**结合多个模型的预测结果：**

```python
# 简单投票
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    estimators=[
        ('nb', nb_model),
        ('lr', lr_model),
        ('svm', svm_model)
    ],
    voting='soft'  # 软投票（基于概率）
)
```

## 9. 课程总结

### 9.1 本节课我们学习了：

**1. 机器学习情感分析的原理**
- 把情感分析看作文本分类问题
- 从训练数据中自动学习规律

**2. 完整的实现流程**
- 数据准备
- 文本预处理
- 特征提取（TF-IDF）
- 模型训练（朴素贝叶斯、逻辑回归、SVM）
- 模型评估

**3. 与词典方法的对比**
- 机器学习方法准确率更高
- 能处理更复杂的语言现象
- 但需要训练数据

**4. 模型优化技巧**
- 增加训练数据
- 改进特征工程
- 调整模型参数
- 使用集成学习

### 9.2 关键要点

- 机器学习方法是情感分析的主流方法
- 数据质量比算法选择更重要
- 需要在训练集和测试集上分别评估
- 可以结合词典方法和机器学习方法

### 9.3 两种方法的选择建议

| 场景 | 推荐方法 | 原因 |
|------|---------|------|
| 快速原型开发 | 词典方法 | 实现简单，不需要训练数据 |
| 有充足训练数据 | 机器学习 | 准确率更高 |
| 需要可解释性 | 词典方法 | 结果易于理解 |
| 追求高准确率 | 机器学习 | 效果更好 |
| 文本表达复杂 | 机器学习 | 能捕捉复杂模式 |
| 领域专用 | 两者结合 | 取长补短 |